In [5]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [6]:
from pathlib import Path
import re
import pandas as pd

# ---------- paths ----------
try:
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    BASE_DIR = Path.cwd()

PROCESSED_DIR = BASE_DIR / "data" / "processed"
UNIFIED_PATH = PROCESSED_DIR / "jobs_unified.csv"
OUT_PATH = PROCESSED_DIR / "jobs_labeled.csv"


# ---------- role patterns (no job_roles_clean.csv used) ----------

ROLE_MAP = {
    # interns / generic SE
    "intern": "INTERN_SE",
    "trainee": "INTERN_SE",
    "graduate": "INTERN_SE",

    # generic software engineer (junior)
    "junior software engineer": "JR_SE",
    "associate software engineer": "JR_SE",
    "software engineer": "JR_SE",

    # frontend
    "frontend developer": "JR_FE_DEV",
    "front end developer": "JR_FE_DEV",
    "front-end developer": "JR_FE_DEV",
    "frontend engineer": "JR_FE_DEV",
    "react developer": "JR_FE_DEV",
    "angular developer": "JR_FE_DEV",

    # backend
    "backend developer": "JR_BE_DEV",
    "back end developer": "JR_BE_DEV",
    "back-end developer": "JR_BE_DEV",
    "backend engineer": "JR_BE_DEV",
    "api developer": "JR_BE_DEV",

    # full-stack
    "full stack developer": "JR_FS_DEV",
    "fullstack developer": "JR_FS_DEV",
    "full-stack developer": "JR_FS_DEV",
    "full stack engineer": "JR_FS_DEV",

    # mobile
    "android developer": "JR_MOBILE_DEV",
    "android engineer": "JR_MOBILE_DEV",
    "mobile developer": "JR_MOBILE_DEV",
    "mobile app developer": "JR_MOBILE_DEV",
    "ios developer": "JR_MOBILE_DEV",
    "ios engineer": "JR_MOBILE_DEV",

    # qa / testing
    "qa engineer": "JR_QA_ENG",
    "quality assurance": "JR_QA_ENG",
    "test engineer": "JR_QA_ENG",
    "software tester": "JR_QA_ENG",
    "qa analyst": "JR_QA_ENG",

    # devops / cloud / sre
    "devops": "DEVOPS_TRAINEE",
    "site reliability": "DEVOPS_TRAINEE",
    "sre": "DEVOPS_TRAINEE",
    "cloud engineer": "DEVOPS_TRAINEE",
    "cloud devops": "DEVOPS_TRAINEE",

    # data / analytics
    "data analyst": "DATA_ANALYST_INT",
    "business intelligence": "DATA_ANALYST_INT",
    "bi analyst": "DATA_ANALYST_INT",

    "data engineer": "DATA_ENGINEER_INT",
    "etl developer": "DATA_ENGINEER_INT",
    "big data engineer": "DATA_ENGINEER_INT",

    # ai / ml
    "machine learning engineer": "AI_ML_ENGINEER_INT",
    "ml engineer": "AI_ML_ENGINEER_INT",
    "ai engineer": "AI_ML_ENGINEER_INT",
    "ai ml engineer": "AI_ML_ENGINEER_INT",

    # it support / sysadmin
    "it support": "JR_IT_SUPPORT",
    "technical support": "JR_IT_SUPPORT",
    "helpdesk": "JR_IT_SUPPORT",
    "service desk": "JR_IT_SUPPORT",
    "support engineer": "JR_IT_SUPPORT",

    "system administrator": "JR_SYS_ADMIN",
    "systems administrator": "JR_SYS_ADMIN",
    "sysadmin": "JR_SYS_ADMIN",
    "systems engineer": "JR_SYS_ADMIN",

    # business / analysis
    "business analyst": "JR_BUSINESS_ANALYST",
    "ba analyst": "JR_BUSINESS_ANALYST",

    # ui/ux / design
    "ui ux designer": "JR_UI_UX_DESIGNER",
    "ui/ux designer": "JR_UI_UX_DESIGNER",
    "ux designer": "JR_UI_UX_DESIGNER",
    "ui designer": "JR_UI_UX_DESIGNER",
}

ROLE_TITLES = {
    "INTERN_SE": "Software Engineering Intern",
    "JR_SE": "Junior Software Engineer",
    "JR_FE_DEV": "Junior Frontend Developer",
    "JR_BE_DEV": "Junior Backend Developer",
    "JR_FS_DEV": "Junior Full-Stack Developer",
    "JR_MOBILE_DEV": "Junior Mobile Developer",
    "JR_QA_ENG": "QA Engineer (Entry Level)",
    "DEVOPS_TRAINEE": "DevOps / Cloud Trainee",
    "DATA_ANALYST_INT": "Data Analyst (Entry Level)",
    "DATA_ENGINEER_INT": "Data Engineer (Entry Level)",
    "AI_ML_ENGINEER_INT": "AI / ML Engineer (Entry Level)",
    "JR_IT_SUPPORT": "IT Support / Helpdesk (Entry Level)",
    "JR_SYS_ADMIN": "System Administrator (Entry Level)",
    "JR_BUSINESS_ANALYST": "Business Analyst (Entry Level)",
    "JR_UI_UX_DESIGNER": "UI/UX Designer (Entry Level)",
}


# ---------- helpers ----------

def normalize_basic(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = re.sub(r"[^a-z0-9+./()\-|@ ]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def clean_title_for_role(raw_title: str) -> str:
    """
    Strip company names, locations, and decorations to get the core job title.
    Examples:
      '(Jr/Sr) QA Engineer, KMS Labs - BONUS' -> 'qa engineer'
      'Java Developer (Mid/Senior) - London' -> 'java developer'
    """
    if not isinstance(raw_title, str):
        return ""

    t = raw_title.strip()

    # keep only part before '@', ' at ', ' - ', '|', ',' (company/location usually after)
    splits = re.split(r"[@|]| - |,|\sat\s", t, maxsplit=1, flags=re.IGNORECASE)
    t = splits[0]

    # remove content inside parentheses
    t = re.sub(r"\([^)]*\)", " ", t)

    # normalize spaces & lowercase
    t = normalize_basic(t)
    return t


def infer_role_id(title_clean: str) -> str:
    for pat, rid in ROLE_MAP.items():
        if pat in title_clean:
            return rid
    return ""


# ---------- main ----------

def main():
    df = pd.read_csv(UNIFIED_PATH, low_memory=False)

    # create cleaned version of title used for role detection
    df["title_clean_for_role"] = df["job_title"].astype(str).apply(clean_title_for_role)

    # infer role_id from cleaned title
    df["role_id"] = df["title_clean_for_role"].apply(infer_role_id)

    # canonical role title
    df["role_title"] = df["role_id"].map(ROLE_TITLES).fillna("")

    # optional: a unified job_title_clean column for modelling/UI
    # if we recognized a role → use canonical title, else use cleaned title
    df["job_title_clean"] = df.apply(
        lambda r: r["role_title"] if r["role_id"] else r["title_clean_for_role"],
        axis=1,
    )

    # keep only rows with a recognized role for training
    labeled_df = df[df["role_id"] != ""].reset_index(drop=True)

    print(f"Labeled {len(labeled_df)} jobs out of {len(df)} total rows")
    labeled_df.to_csv(OUT_PATH, index=False)
    print(f"Saved labeled dataset to {OUT_PATH}")


if __name__ == "__main__":
    main()


Labeled 5470 jobs out of 15230 total rows
Saved labeled dataset to /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/processed/jobs_labeled.csv
